This report predicts the probability that a borrower will experience financial distress within the next two years, using 250,000 historical records pre-split into training (`cs-training.csv`) and test (`cs-test.csv`) sets. The analysis proceeds through a structured diagnostic phase: profiling each feature, visually and mathematically verifying non-normality, scoring skewness to determine pipeline routing, and selecting the correct power transformer per feature.

In [ ]:
#| echo: false
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import normaltest
from great_tables import GT

SEED: int = 42
TARGET: str = "SeriousDlqin2yrs"

random.seed(SEED)
np.random.seed(SEED)

df_train = pd.read_csv("data/cs-training.csv", index_col=0)
df_test  = pd.read_csv("data/cs-test.csv",  index_col=0)

features: list[str] = [c for c in df_train.columns if c != TARGET]

## Missing Value Profiling

Algorithms like SVM and MLP will throw fatal errors on a single `NaN`, so the first step is to quantify any missingness in the training set before any transformation is applied. If missing values are found, understanding the mechanism behind them determines the correct treatment. When data is missing completely at random (MCAR), the gap is unrelated to the underlying values and median imputation is safe. When data is missing not at random (MNAR) — for instance, an unemployed borrower omitting `MonthlyIncome` — the missingness is caused by the value itself, and median imputation introduces bias. In that case, a binary indicator column or model-based imputation is required, and the imputation step must be chained before any downstream mathematical transformation to prevent errors.

In [ ]:
#| echo: false
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a column-level summary: dtype, null count, null %, and basic stats.

    Args:
        df: The DataFrame to profile.

    Returns:
        Summary DataFrame with one row per column.
    """
    null_count = df.isnull().sum()
    null_pct   = (null_count / len(df) * 100).round(2)
    return pd.DataFrame({
        "dtype"     : df.dtypes,
        "null_count": null_count,
        "null_%"    : null_pct,
        "mean"      : df.mean(numeric_only=True).round(4),
        "std"       : df.std(numeric_only=True).round(4),
        "min"       : df.min(numeric_only=True),
        "max"       : df.max(numeric_only=True),
    })


GT(profile_dataframe(df_train).reset_index().rename(columns={"index": "feature"}))

In [ ]:
#| echo: false
def test_mcar(df: pd.DataFrame, cols: list[str], target: str) -> pd.DataFrame:
    """Test whether missingness in each column is associated with the target variable.

    A statistically significant point-biserial correlation between a missing
    indicator and the target is evidence against MCAR, suggesting MAR or MNAR.

    Args:
        df: Training DataFrame.
        cols: Columns to test for informative missingness.
        target: Binary target column name.

    Returns:
        DataFrame with default rates, correlation, and p-value per column.
    """
    from scipy.stats import pointbiserialr

    rows = []
    for col in cols:
        indicator = df[col].isna().astype(int)
        r, p = pointbiserialr(indicator, df[target])
        rows.append({
            "column": col,
            "missing_pct": round(indicator.mean() * 100, 2),
            "default_rate_observed": round(df.loc[indicator == 0, target].mean(), 4),
            "default_rate_missing": round(df.loc[indicator == 1, target].mean(), 4),
            "r": round(r, 4),
            "p_value": round(p, 10),
        })
    return pd.DataFrame(rows)

missing_cols = [c for c in features if df_train[c].isna().any()]
mcar_df = test_mcar(df_train, missing_cols, TARGET)
(
    GT(mcar_df)
    .tab_header(
        title="MCAR Test — Point-Biserial Correlation",
        subtitle="Is missingness associated with the target variable?"
    )
    .fmt_number(columns=["missing_pct", "default_rate_observed", "default_rate_missing", "r"], decimals=4)
    .fmt_scientific(columns="p_value")
)

Both columns return p-values far below 0.05, so we reject MCAR for both. Missingness is statistically associated with the target variable, confirming MNAR.

What makes this a genuine surprise is the direction: rows with missing values show a lower default rate, not higher. The intuitive assumption would be the opposite — that missing income signals financial instability and therefore higher risk. Instead, the data suggests the missing group may skew toward retired borrowers: no employment income to report, but also lower credit risk. The absence of a value is itself a meaningful signal, just not the one we expected.

Median imputation alone would silently discard this signal. The correct treatment is to first create a binary `_is_missing` flag for each affected column — preserving the information that absence of a value is predictive — and then impute the median to satisfy algorithm requirements.

We now apply the two-step fix derived above, creating the binary flags before imputation so no signal is overwritten by the fill.

In [ ]:
#| echo: false
MONTHLY_INCOME_MEDIAN: float = df_train["MonthlyIncome"].median()
DEPENDENTS_MEDIAN: float = df_train["NumberOfDependents"].median()


def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean missing-value indicator columns before imputation.

    Flags are derived before filling so the signal is not overwritten.
    Uses integer encoding (0/1) for direct compatibility with sklearn estimators.

    Args:
        df: DataFrame to annotate.

    Returns:
        DataFrame with IsMonthlyIncomeMissing and IsNumberOfDependentsMissing appended.
    """
    return df.assign(
        IsMonthlyIncomeMissing=df["MonthlyIncome"].isna().astype(int),
        IsNumberOfDependentsMissing=df["NumberOfDependents"].isna().astype(int),
    )


def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values using training-set medians.

    MonthlyIncome is filled with the training median.
    NumberOfDependents is filled with 0, which equals the training median
    (59% of observed values are 0) and aligns with the retiree hypothesis.

    Args:
        df: DataFrame with missing-value flag columns already added.

    Returns:
        DataFrame with NaNs in MonthlyIncome and NumberOfDependents resolved.
    """
    return df.fillna({
        "MonthlyIncome": MONTHLY_INCOME_MEDIAN,
        "NumberOfDependents": DEPENDENTS_MEDIAN,
    })


df_train = impute_missing(add_missing_flags(df_train))
features = [c for c in df_train.columns if c != TARGET]
df_train[["MonthlyIncome", "IsMonthlyIncomeMissing",
          "NumberOfDependents", "IsNumberOfDependentsMissing"]].describe().T

## Class Imbalance — Target Variable

We examine the distribution of `SeriousDlqin2yrs` to understand how the two outcomes are represented in the training set. A large disparity between classes means a naive model could achieve high accuracy by always predicting the majority class, while having zero predictive power for actual defaults. If severe imbalance is detected, two remedies are available. SMOTE synthesises minority-class examples in the training fold to balance the class ratio, while passing `class_weight='balanced'` to sklearn estimators weights the minority class inversely proportional to its frequency — a lower-cost option that requires no synthetic data. Either strategy must be applied only on the training set to prevent leakage into the test evaluation.

In [ ]:
#| echo: false
def check_class_balance(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """Compute class distribution for the target variable.

    Args:
        df: Training DataFrame.
        target: Name of the target column.

    Returns:
        DataFrame with class label, count, and percentage.
    """
    counts = df[target].value_counts().rename_axis("class").reset_index(name="count")
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts

class_balance_df = check_class_balance(df_train, TARGET)
(
    GT(class_balance_df)
    .tab_header(title="Class Distribution", subtitle=TARGET)
    .fmt_integer(columns="count")
    .fmt_number(columns="pct", decimals=2)
)

## Distribution Analysis

### Visual Check

Histograms for every continuous feature give us a first look at each feature's shape before any formal testing. We are looking for heavy right tails, narrow spikes at zero, or multi-modal shapes — any of which indicate deviation from normality that could destabilise distance-based and gradient-descent models.

In [ ]:
#| echo: false
df_train[features].hist(bins=60, figsize=(15, 10), layout=(4, 3))
plt.suptitle("Feature Distributions — Training Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Mathematical Proof (D'Agostino K²)

To formally test whether each feature follows a normal distribution, we apply D'Agostino's $K^2$ test, which combines skewness and kurtosis into a single statistic. The null hypothesis $H_0$ is that the feature originates from a normal distribution. A p-value below 0.05 gives grounds to reject $H_0$, confirming the feature is statistically non-normal. Features where we fail to reject may not require power transformation. If non-normality is widespread, standard scaling alone will be insufficient and the pipeline must rely on power transformations to address the underlying distribution shape.

In [ ]:
#| echo: false
def run_normality_tests(df: pd.DataFrame, features: list[str], alpha: float = 0.05) -> pd.DataFrame:
    """Run D'Agostino's K² normality test on each feature and report results.

    Args:
        df:       DataFrame containing the features.
        features: Column names to test.
        alpha:    Significance level for rejecting H0 (default 0.05).

    Returns:
        DataFrame with columns: feature, statistic, p_value, reject_H0, verdict.
    """
    records = [
        {
            "feature"  : col,
            "statistic": round(stat, 4),
            "p_value"  : round(p, 6),
            "reject_H0": p < alpha,
            "verdict"  : "Non-normal" if p < alpha else "Normal",
        }
        for col in features
        for stat, p in [normaltest(df[col].dropna())]
    ]
    return pd.DataFrame(records)


GT(run_normality_tests(df_train, features))

### Data Quality — DebtRatio Encoding Inconsistency

`DebtRatio` is defined as monthly debt payments divided by monthly income. If `MonthlyIncome` is missing, a valid ratio should be uncomputable — yet every row with missing income still carries a non-null `DebtRatio`. We examine whether the column is encoding something different for those rows by comparing its distribution between the two groups. A large divergence would suggest the column has mixed semantics.

In [ ]:
#| echo: false
def compare_groups_by_missingness(
    df: pd.DataFrame, flag_col: str, compare_features: list[str]
) -> pd.DataFrame:
    """Compare feature medians between rows where flag_col is 1 vs 0.

    Args:
        df: Training DataFrame.
        flag_col: Binary indicator column (1 = was missing, 0 = was present).
        compare_features: Features to summarise across the two groups.

    Returns:
        DataFrame with per-feature medians for each group and their ratio.
    """
    is_missing = df[flag_col].astype(bool)
    rows = []
    for col in compare_features:
        med_present = df.loc[~is_missing, col].median()
        med_missing = df.loc[is_missing, col].median()
        ratio = (
            round(med_missing / med_present, 2)
            if med_present not in (0, None) and pd.notna(med_present) and pd.notna(med_missing)
            else None
        )
        rows.append({
            "feature": col,
            "median_income_present": med_present,
            "median_income_missing": med_missing,
            "ratio": ratio,
        })
    return (
        pd.DataFrame(rows)
        .sort_values("ratio", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

non_target_features = [c for c in df_train.columns if c != TARGET]
group_df = compare_groups_by_missingness(df_train, "IsMonthlyIncomeMissing", non_target_features)
(
    GT(group_df)
    .tab_header(
        title="Feature Medians — MonthlyIncome Missing vs. Present",
        subtitle="Sorted by ratio (missing / present) descending — ratio column omitted where present median is 0"
    )
    .fmt_number(columns=["median_income_present", "median_income_missing", "ratio"], decimals=4)
)

In [ ]:
#| echo: false
def plot_debtratio_before_after(df: pd.DataFrame, flag_col: str, col: str = "DebtRatio") -> None:
    """Plot DebtRatio distribution before and after isolating the corrupted rows.

    Args:
        df: Training DataFrame with binary missing-income flag.
        flag_col: Binary indicator for missing income (1 = missing).
        col: Feature column to plot.
    """
    is_corrupted = df[flag_col].astype(bool)
    before = df[col]
    after  = df.loc[~is_corrupted, col]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        "DebtRatio — Before and After Removing Corrupted Rows",
        fontsize=12, fontweight="bold"
    )

    axes[0].hist(before.clip(upper=before.quantile(0.995)), bins=80, color="steelblue", alpha=0.8)
    axes[0].set_title("All rows (clipped at 99.5th percentile for visibility)")
    axes[0].set_xlabel("DebtRatio")
    axes[0].set_ylabel("count")

    axes[1].hist(after.clip(upper=5), bins=80, color="darkorange", alpha=0.8)
    axes[1].set_title("Income-present rows only (clipped at 5 for visibility)")
    axes[1].set_xlabel("DebtRatio")

    plt.tight_layout()
    plt.show()


plot_debtratio_before_after(df_train, "IsMonthlyIncomeMissing")

`DebtRatio` is the only feature with a pathological divergence — its median is roughly 3,900 times higher in the missing-income group (1,159 vs 0.296). While a debt-to-income ratio above 1.0 is technically possible, values in the hundreds or thousands are not interpretable as any ratio. The 75th percentile sits at 0.87, then the 90th jumps to 1,267 — that is not a heavy tail, it is two irreconcilable encodings in the same column. For the missing-income group, the column almost certainly stores raw monthly debt obligations in dollars rather than a computed ratio.

The broader pattern reinforces the retired-borrower hypothesis: the missing-income group is older (median age 57 vs 51) and carries lower revolving utilisation (0.08 vs 0.18). Since `IsMonthlyIncomeMissing` already captures the structural difference of these rows, the cleanest action is to drop `DebtRatio` entirely — removing a corrupted feature without discarding any information not already represented more reliably elsewhere.

### Data Quality — RevolvingUtilization Extreme Values

`RevolvingUtilizationOfUnsecuredLines` should be a ratio between 0 and 1, representing credit used divided by credit limit. Values marginally above 1.0 are plausible, since fees and interest can push balances past the stated limit, but we inspect the upper tail for the same dollar-encoding issue found in `DebtRatio`. A cliff in the percentile distribution — where the column transitions from ratio-scale values to implausibly large numbers — would confirm the problem.

In [ ]:
#| echo: false
def profile_upper_tail(
    df: pd.DataFrame, col: str, percentiles: list[float] | None = None
) -> pd.DataFrame:
    """Summarise the upper tail of a column to surface encoding discontinuities.

    Args:
        df: Training DataFrame.
        col: Column to profile.
        percentiles: Percentile breakpoints to include. Defaults to standard set.

    Returns:
        DataFrame of percentile labels and their corresponding values.
    """
    pcts = percentiles or [0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.0]
    labels = [f"p{int(p * 100)}" if p < 1 else "max" for p in pcts]
    values = df[col].quantile(pcts).values
    return pd.DataFrame({"percentile": labels, "value": values})

tail_df = profile_upper_tail(df_train, "RevolvingUtilizationOfUnsecuredLines")
(
    GT(tail_df)
    .tab_header(
        title="RevolvingUtilizationOfUnsecuredLines — Upper Tail",
        subtitle="Identifying the encoding cliff"
    )
    .fmt_number(columns="value", decimals=4)
)

The cliff is unmistakable. The 90th percentile is 0.98, the 95th is 1.0, and the 99.9th jumps to 1,571 with a maximum of 50,708. Values marginally above 1.0 are defensible, but values in the thousands are dollar balances, not ratios — the same encoding inconsistency found in `DebtRatio`. Unlike that column, however, the feature is well-behaved for the vast majority of rows, so dropping it would discard real signal. Capping at the 99th percentile (~1.09) preserves legitimate near-limit values while suppressing the dollar-encoded outliers.

### Data Quality — Delinquency Column Error Codes

The three delinquency count columns showed a suspicious gap in their value distributions — counts jump from plausible values (0–17) directly to 96 and 98, with nothing in between. In financial survey datasets, values such as 96, 98, and 99 are commonly used as error codes to flag records as not applicable or erroneous rather than genuine counts. We investigate whether the same rows carry these anomalous values across all three columns simultaneously, which would confirm they are error codes rather than real delinquency counts.

In [ ]:
#| echo: false
def find_error_code_rows(
    df: pd.DataFrame, cols: list[str], error_codes: list[int]
) -> pd.DataFrame:
    """Identify rows carrying error codes and verify cross-column consistency.

    Args:
        df: Training DataFrame.
        cols: Columns expected to share the error code encoding.
        error_codes: Candidate error codes to check.

    Returns:
        DataFrame summarising error code counts and cross-column consistency.
    """
    rows = []
    for val in error_codes:
        masks = [df[c] == val for c in cols]
        count = masks[0].sum()
        all_identical = all((masks[0] == m).all() for m in masks[1:])
        rows.append({
            "error_code": val,
            "row_count": int(count),
            "identical_across_all_columns": all_identical,
        })
    return pd.DataFrame(rows)

delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
error_code_df = find_error_code_rows(df_train, delinquency_cols, error_codes=[96, 98])
(
    GT(error_code_df)
    .tab_header(
        title="Delinquency Column Error Codes",
        subtitle="Do the same rows carry the same anomalous value across all three columns?"
    )
    .fmt_integer(columns=["error_code", "row_count"])
)

Both error codes (96 and 98) are carried by exactly the same rows across all three delinquency columns. A real borrower cannot have identical counts of 30–59, 60–89, and 90+ day delinquencies, since these are sequential severity buckets. This conclusively identifies them as error codes. With only 269 affected rows representing 0.18% of the training set, the cleanest treatment is to replace these values with `NaN` and drop the affected rows outright — imputing delinquency counts for corrupted records would risk introducing more noise than signal.

### Data Quality — MonthlyIncome Extreme Values

`MonthlyIncome` has a maximum of \$3,008,750 per month, roughly 38 times the 99.9th percentile. Before accepting these as legitimate, we check whether extreme income values are associated with lower default rates and whether they are concentrated enough to distort downstream transformations. We are looking both for a clear relationship between income and default risk and for a distributional cliff similar to those found in `DebtRatio` and `RevolvingUtilization`.

In [ ]:
#| echo: false
def default_rate_by_income_bucket(
    df: pd.DataFrame, income_col: str, target: str
) -> pd.DataFrame:
    """Compute default rate, row count, and median income across income brackets.

    Args:
        df: Training DataFrame.
        income_col: Name of the income column.
        target: Binary target column name.

    Returns:
        DataFrame with one row per income bracket.
    """
    df_valid = df.dropna(subset=[income_col]).copy()
    df_valid["income_bucket"] = pd.cut(
        df_valid[income_col],
        bins=[0, 2_000, 5_000, 10_000, 20_000, 50_000, float("inf")],
        labels=["<$2k", "$2k–$5k", "$5k–$10k", "$10k–$20k", "$20k–$50k", ">$50k"],
    )
    return (
        df_valid.groupby("income_bucket", observed=True)
        .agg(
            row_count=(income_col, "count"),
            median_income=(income_col, "median"),
            default_rate=(target, "mean"),
        )
        .round(4)
        .reset_index()
    )

income_bucket_df = default_rate_by_income_bucket(df_train, "MonthlyIncome", TARGET)
(
    GT(income_bucket_df)
    .tab_header(
        title="Default Rate by Monthly Income Bracket",
        subtitle="Rows with missing MonthlyIncome excluded"
    )
    .fmt_integer(columns="row_count")
    .fmt_currency(columns="median_income", currency="USD")
    .fmt_percent(columns="default_rate", decimals=2)
)

Higher income does reduce default risk, but not monotonically. The default rate falls sharply from 9.1% at under \$2k per month to 4.2% at \$10k–\$20k, then slightly increases for the highest earners (5.3% at \$20k–\$50k, 5.7% above \$50k). Very high earners may carry more leverage or take on riskier debt structures, so the assumption that wealth implies safety does not hold universally at the extreme end.

The extreme values are almost certainly real rather than encoding errors — they are demographically coherent and all non-defaulting. However, a handful of values 38 times the 99.9th percentile will dominate the Yeo-Johnson lambda fit for Branch B models. For tree models in Branch A, no cap is needed since trees split on thresholds and handle extreme values natively. For Branch B, `MonthlyIncome` is capped at the 99.9th percentile (~\$78k/month) before transformation, applied only after the `IsMonthlyIncomeMissing` flag and imputation steps are complete.

## Multicollinearity — Correlation Check

We compute the Pearson correlation matrix across all features and inspect the top 10 most correlated pairs. Highly collinear pairs are measuring the same underlying signal, so retaining both adds noise without adding information. Pairs with $|r| > 0.8$ are candidates for removal. For SVM and MLP, redundant features cause computational bloat and increase overfitting risk. For tree models, feature importance scores become artificially diluted across collinear features, rendering them uninterpretable. The appropriate response is to drop the redundant feature from each flagged pair, retaining the more informative signal.

In [ ]:
#| echo: false
def top_correlations(
    df: pd.DataFrame, features: list[str], n: int = 10
) -> pd.DataFrame:
    """Return the top N most correlated feature pairs by absolute Pearson r.

    Args:
        df: Training DataFrame.
        features: Feature columns to evaluate.
        n: Number of top pairs to return.

    Returns:
        DataFrame of feature pairs sorted by absolute correlation, descending.
    """
    corr = df[features].corr(method="pearson")
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["feature_a", "feature_b", "correlation"]
    return (
        pairs.reindex(pairs["correlation"].abs().sort_values(ascending=False).index)
        .head(n)
        .reset_index(drop=True)
    )

top_corr_df = top_correlations(df_train, features)
(
    GT(top_corr_df)
    .tab_header(title="Top 10 Feature Correlations", subtitle="Pearson r — sorted by absolute value")
    .fmt_number(columns="correlation", decimals=4)
)

## Skewness Heuristic — Pipeline Branching

To programmatically route features into the correct pipeline branch, we compute the absolute skewness score for each feature and apply a threshold rule: features with $|\text{skew}| > 1$ are considered highly skewed and routed to Branch B for transformation and scaling, while those at or below the threshold are considered approximately symmetric and routed to Branch A to be passed raw and unscaled. Features with low skew scores can safely bypass power transformation; those above the threshold require treatment before distance-based or gradient-descent models can use them effectively.

In [ ]:
#| echo: false
def compute_skewness_table(df: pd.DataFrame, features: list[str], threshold: float = 1.0) -> pd.DataFrame:
    """Compute skewness for each feature and assign a pipeline branch.

    Args:
        df:        DataFrame containing the features.
        features:  Column names to evaluate.
        threshold: Absolute skew above which a feature is flagged (default 1.0).

    Returns:
        DataFrame sorted by abs_skew descending.
    """
    records = [
        {
            "feature"      : col,
            "skewness"     : round(df[col].skew(), 4),
            "abs_skew"     : round(abs(df[col].skew()), 4),
            "highly_skewed": abs(df[col].skew()) > threshold,
            "branch"       : "B — Transform + Scale" if abs(df[col].skew()) > threshold else "A — Raw (tree-safe)",
        }
        for col in features
    ]
    return pd.DataFrame(records).sort_values("abs_skew", ascending=False)


skewness_df = compute_skewness_table(df_train, features)
GT(skewness_df)

## Transformer Selection — Box-Cox vs. Yeo-Johnson

Power transformers have strict domain requirements, so before applying any transformation we inspect the minimum value of each Branch B feature to select the correct algorithm. The decision rule is: if $\min(\text{feature}) > 0$, use Box-Cox; if $\min(\text{feature}) \le 0$, use Yeo-Johnson.

While Yeo-Johnson safely handles negatives and zeros, defaulting to it unconditionally carries real trade-offs. It optimises a complex piecewise function that is slower on large datasets. Box-Cox lambdas map to interpretable classic transforms — $\lambda = 0$ is exactly a log transform — whereas Yeo-Johnson's $+1$ shift obscures the underlying adjustment. For strictly positive data, Box-Cox also tends to find a mathematically tighter normal fit. The domain check here is not a formality: it determines the strictly correct algorithm.

In [ ]:
#| echo: false
def select_transformer(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Assign a power transformer to each feature based on its minimum value.

    Decision rule:
      min > 0  + Box-Cox (strictly positive data)
      min <= 0 + Yeo-Johnson (handles zeros and negatives)

    Args:
        df:       DataFrame containing the features.
        features: Column names to evaluate (typically the highly-skewed subset).

    Returns:
        DataFrame with columns: feature, min_value, transformer, reason.
    """
    records = [
        {
            "feature"    : col,
            "min_value"  : df[col].min(),
            "transformer": "Box-Cox" if df[col].min() > 0 else "Yeo-Johnson",
            "reason"     : "min > 0 — strictly positive" if df[col].min() > 0
                           else "min ≤ 0 — contains zeros or negatives",
        }
        for col in features
    ]
    return pd.DataFrame(records)


highly_skewed_features = skewness_df.loc[skewness_df["highly_skewed"], "feature"].tolist()
transformer_plan = select_transformer(df_train, highly_skewed_features)
GT(transformer_plan)

## Pipeline

The diagnosis established four data quality treatments and a clear branching strategy, all of which are encoded here into a reproducible sklearn pipeline fitted strictly on the training set. For both branches, error codes (96 and 98) are replaced with `NaN` in the delinquency columns and the affected rows dropped, `DebtRatio` is dropped due to its corrupted dollar-amount encoding, the two delinquency columns with near-perfect collinearity with `NumberOfTimes90DaysLate` are dropped, and `RevolvingUtilizationOfUnsecuredLines` is capped at the 99.9th percentile. Branch A passes the cleaned features through raw and unscaled for the tree models. Branch B additionally caps `MonthlyIncome` at the 99.9th percentile before applying Yeo-Johnson followed by StandardScaler to all skewed features.

In [ ]:
#| echo: true
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer, StandardScaler
from typing import Protocol
from sklearn.base import BaseEstimator
from enum import Enum, auto


class FeatureTransformer(Protocol):
    """Interface for per-feature Branch B transformations."""

    def __str__(self) -> str: ...
    def pipeline(self) -> BaseEstimator: ...


class BinaryPassthrough(FeatureTransformer):
    """Identity transform for binary indicator features — preserves 0/1 encoding unchanged."""

    def __str__(self) -> str:
        return "passthrough"

    def pipeline(self) -> FunctionTransformer:
        return FunctionTransformer()


class SymmetricTransformer(FeatureTransformer):
    """Standard scaling only, for low-skew continuous features that need centering but not reshaping."""

    def __str__(self) -> str:
        return "StandardScaler"

    def pipeline(self) -> StandardScaler:
        return StandardScaler()


class SkewedTransformer(FeatureTransformer):
    """Yeo-Johnson followed by standard scaling, for highly skewed features without extreme outliers."""

    def __str__(self) -> str:
        return "Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", StandardScaler()),
        ])


class CappedSkewedTransformer(SkewedTransformer):
    """Extends SkewedTransformer by prepending a percentile cap for features with extreme outliers.

    Args:
        cap: Upper bound to clip values to before the Yeo-Johnson transform.
    """

    def __init__(self, *, cap: float) -> None:
        self.cap = cap

    def __str__(self) -> str:
        return "cap \u2192 Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("cap", FunctionTransformer(lambda X: np.clip(X, None, self.cap))),
            *super().pipeline().steps,
        ])




REVOLVING_CAP: float = df_train["RevolvingUtilizationOfUnsecuredLines"].quantile(0.999)
INCOME_CAP: float = df_train["MonthlyIncome"].quantile(0.999)
FEATURE_PIPELINE_MAP: dict[str, FeatureTransformer] = {
    "RevolvingUtilizationOfUnsecuredLines": CappedSkewedTransformer(cap=REVOLVING_CAP),
    "age":                                  SymmetricTransformer(),
    "MonthlyIncome":                        CappedSkewedTransformer(cap=INCOME_CAP),
    "NumberOfOpenCreditLinesAndLoans":      SkewedTransformer(),
    "NumberOfTimes90DaysLate":              SkewedTransformer(),
    "NumberRealEstateLoansOrLines":         SkewedTransformer(),
    "NumberOfDependents":                   SkewedTransformer(),
    "IsMonthlyIncomeMissing":               BinaryPassthrough(),
    "IsNumberOfDependentsMissing":          BinaryPassthrough(),
}

df_clean = (
    df_train
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(96, np.nan))
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(98, np.nan))
    .dropna(subset=["NumberOfTimes90DaysLate"])
    .reset_index(drop=True)
)

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"feature": col, "branch_a": "raw", "branch_b": str(tx)}
        for col, tx in FEATURE_PIPELINE_MAP.items()
    ]))
    .tab_header(title="Feature Routing", subtitle="Treatment applied per branch")
)

## Train / Validation Split

The test set labels are not publicly available — evaluating against them requires uploading predictions through the Kaggle API, which is outside the scope of this local analysis. To measure model performance without submitting, we reserve 20% of the training data as a held-out validation set.

Because the target is heavily imbalanced (roughly 6.7% default rate), the split is stratified: passing `stratify=_y` to `train_test_split` ensures both halves retain the original class ratio. Without stratification, random sampling could accidentally concentrate defaults into one partition and produce misleading performance estimates.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.base import TransformerMixin
from dataclasses import dataclass

VAL_FRAC: float        = 0.20
_X:       pd.DataFrame = df_clean[list(FEATURE_PIPELINE_MAP.keys())]
_y:       pd.Series    = df_clean[TARGET]


@dataclass(eq=False)
class Split:
    """Feature matrix and target vector for one dataset partition."""
    X: pd.DataFrame
    y: pd.Series


class Dataset:
    """Training and validation splits for one preprocessing branch."""

    def __init__(
        self,
        raw_X: pd.DataFrame,
        raw_y: pd.Series,
        *,
        test_size: float = 0.2,
        random_state: int | None = None,
        stratify: pd.Series | None = None,
    ) -> None:
        """Split raw_X and raw_y and construct a Dataset.

        Args:
            raw_X: Full feature DataFrame before splitting.
            raw_y: Full target Series before splitting.
            test_size: Fraction of rows reserved for the validation split.
            random_state: Random seed forwarded to train_test_split.
            stratify: Series passed to train_test_split to preserve class balance.
        """
        X_tr, X_val, y_tr, y_val = train_test_split(raw_X, raw_y, test_size=test_size, random_state=random_state, stratify=stratify)
        self.train = Split(X=X_tr.reset_index(drop=True),  y=y_tr.reset_index(drop=True))
        self.val   = Split(X=X_val.reset_index(drop=True), y=y_val.reset_index(drop=True))

    @classmethod
    def pipelined(
        cls,
        raw_X: pd.DataFrame,
        raw_y: pd.Series,
        *,
        pipeline: TransformerMixin,
        test_size: float = 0.2,
        random_state: int | None = None,
        stratify: pd.Series | None = None,
    ) -> "Dataset":
        """Split raw_X and raw_y, apply pipeline, then delegate to __init__.

        Args:
            raw_X: Full feature DataFrame before splitting.
            raw_y: Full target Series before splitting.
            pipeline: sklearn TransformerMixin fitted on the training half
                and applied to both halves — accepts any TransformerMixin
                (ColumnTransformer, Pipeline, etc.).
            test_size: Fraction of rows reserved for the validation split.
            random_state: Random seed forwarded to train_test_split.
            stratify: Series passed to train_test_split to preserve class balance.

        Returns:
            Dataset with .train and .val Split objects populated.
        """
        dataset = cls(raw_X, raw_y, test_size=test_size, random_state=random_state, stratify=stratify)
        cols = list(raw_X.columns)
        X_tr_pipelined  = pd.DataFrame(pipeline.fit_transform(dataset.train.X), columns=cols)
        X_val_pipelined = pd.DataFrame(pipeline.transform(dataset.val.X),       columns=cols)
        dataset.train.X = X_tr_pipelined
        dataset.val.X   = X_val_pipelined
        return dataset


class Data(Enum):
    """Dataset variants keyed by preprocessing branch.

    Each member's value is a Dataset with .train and .val Split objects.
    Split.X is the feature matrix; Split.y is the target vector.
    All preprocessors are fitted on .train only; .val is never seen during fitting.
    """
    CLEAN  = Dataset(_X, _y, test_size=VAL_FRAC, random_state=SEED, stratify=_y)
    SCALED = Dataset.pipelined(
        _X, _y,
        pipeline=ColumnTransformer(
            transformers=[(col, tx.pipeline(), [col]) for col, tx in FEATURE_PIPELINE_MAP.items()],
            remainder="drop",
        ),
        test_size=VAL_FRAC, random_state=SEED, stratify=_y,
    )

    @property
    def train(self) -> Split:
        return self.value.train

    @property
    def val(self) -> Split:
        return self.value.val

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"split": "train", "branch": "A — Tree models", "rows": Data.CLEAN.train.X.shape[0],  "features": Data.CLEAN.train.X.shape[1],  "transformations": "none (raw)"},
        {"split": "val",   "branch": "A — Tree models", "rows": Data.CLEAN.val.X.shape[0],    "features": Data.CLEAN.val.X.shape[1],    "transformations": "none (raw)"},
        {"split": "train", "branch": "B — SVM / MLP",   "rows": Data.SCALED.train.X.shape[0], "features": Data.SCALED.train.X.shape[1], "transformations": "per feature routing"},
        {"split": "val",   "branch": "B — SVM / MLP",   "rows": Data.SCALED.val.X.shape[0],   "features": Data.SCALED.val.X.shape[1],   "transformations": "per feature routing"},
    ]))
    .tab_header(title="Pipeline Output", subtitle="Dataset dimensions per branch and split")
    .fmt_integer(columns=["rows", "features"])
)

In [ ]:
#| echo: false
(
    GT(pd.DataFrame([
        {"branch": "A — Tree models", "rows": Data.CLEAN.train.X.shape[0],  "features": Data.CLEAN.train.X.shape[1],  "transformations": "none (raw)"},
        {"branch": "B — SVM / MLP",   "rows": Data.SCALED.train.X.shape[0], "features": Data.SCALED.train.X.shape[1], "transformations": "per feature routing"},
    ]))
    .tab_header(title="Pipeline Output", subtitle="Training set dimensions per branch")
    .fmt_integer(columns=["rows", "features"])
)

In [ ]:
#| echo: false
def plot_branch_b_distributions(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    features: list[str],
) -> None:
    """Plot Branch B feature distributions before and after pipeline transformation.

    Args:
        df_before: Cleaned but untransformed feature DataFrame.
        df_after: Transformed Branch B DataFrame.
        features: Features to compare — should be the skewed Branch B columns.
    """
    n = len(features)
    fig, axes = plt.subplots(n, 2, figsize=(14, n * 2.5))
    fig.suptitle(
        "Branch B Feature Distributions — Before vs. After Pipeline",
        fontsize=13, fontweight="bold", y=1.01,
    )
    for i, feat in enumerate(features):
        axes[i, 0].hist(df_before[feat].dropna(), bins=60, color="steelblue", alpha=0.8)
        axes[i, 0].set_title(f"{feat}  |  before", fontsize=9)
        axes[i, 0].set_ylabel("count")

        axes[i, 1].hist(df_after[feat], bins=60, color="darkorange", alpha=0.8)
        axes[i, 1].set_title(f"{feat}  |  after Yeo-Johnson + StandardScaler", fontsize=9)

    plt.tight_layout()
    plt.show()


transformed_features = [col for col, tx in FEATURE_PIPELINE_MAP.items() if isinstance(tx, SkewedTransformer)]
plot_branch_b_distributions(Data.CLEAN.train.X, Data.SCALED.train.X, transformed_features)

## Summary

The sections above establish a complete diagnostic picture of the training data. Features with $|\text{skew}| \le 1$ are passed raw and unscaled to the tree models in Branch A. Highly skewed features are imputed, power-transformed, and scaled for the distance and gradient-based models in Branch B. All imputers, transformers, and scalers are fitted only on the training set and applied via `.transform()` to the test set — `.fit()` and `.fit_transform()` are never called on test data.

## Model Evaluation Framework

Eight models are evaluated against the held-out validation set across the two preprocessing branches. Each has its own hyperparameter search space, but the evaluation logic is identical: tune on a subsampled training split, extract the best estimator, and score it against the held-out validation split.

| Model | Features | Estimator |
|---|---|---|
| Naive Baseline | Raw | — |
| SVM — Hard margin | Raw | `SVC` |
| SVM — Hard margin | Scaled | `SVC` |
| SVM — Soft margin | Scaled | `SVC` |
| Logistic Regression | Scaled | `LogisticRegression` |
| Gradient Boosting | Raw | `GradientBoostingClassifier` |
| Random Forest | Raw | `RandomForestClassifier` |
| Decision Tree | Raw | `DecisionTreeClassifier` |
| MLP | Scaled | `MLPClassifier` |

To avoid duplicating evaluation scaffolding for every model, we define a `Model` abstract base class. Subclasses declare three things and nothing else:

- `label` — display name used in tables and plots
- `data` — which preprocessing branch (`Data.CLEAN` or `Data.SCALED`)
- `_build_search()` — returns an unfitted `RandomizedSearchCV` configured for that algorithm, or `None` for the naive baseline

The base class seals the rest: subsampling the training split, fitting the search, extracting `best_estimator_`, computing predicted probabilities, and deriving the ROC curve and AUC. All of these are lazy `cached_property` values — nothing runs until first accessed.

In [ ]:
from abc import ABC, abstractmethod
from functools import cached_property
from typing import ClassVar
from sklearn.base import BaseEstimator
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, roc_curve

FAST_MODE: bool = True   # True while iterating; False for final evaluation

TUNE_SAMPLE: int = 3_000  if FAST_MODE else 10_000
TUNE_ITER:   int = 5      if FAST_MODE else 20
TUNE_FOLDS:  int = 2      if FAST_MODE else 3


class Model(ABC):
    """Base class for all models in the evaluation pipeline.

    Subclasses declare label, data, and _build_search.
    All evaluation metrics are computed lazily and cached on first access.
    """

    label: ClassVar[str]
    data:  ClassVar[Data]

    @abstractmethod
    def _build_search(self) -> RandomizedSearchCV | None:
        """Return an unfitted RandomizedSearchCV, or None for the naive baseline.

        Returns:
            Unfitted RandomizedSearchCV, or None to fall back to constant scores.
        """
        ...

    @cached_property
    def search(self) -> RandomizedSearchCV | None:
        """Fit the search on a subsampled training split.

        Returns:
            Fitted RandomizedSearchCV, or None if _build_search returns None.
        """
        s = self._build_search()
        if s is None:
            return None
        idx = np.random.RandomState(SEED).choice(
            len(self.data.train.X), size=min(TUNE_SAMPLE, len(self.data.train.X)), replace=False,
        )
        s.fit(self.data.train.X.iloc[idx], self.data.train.y.iloc[idx])
        return s

    @cached_property
    def estimator(self) -> BaseEstimator:
        """Best estimator from the fitted search."""
        return self.search.best_estimator_

    @cached_property
    def scores(self) -> np.ndarray:
        """Predicted probabilities on the validation split."""
        return self.estimator.predict_proba(self.data.val.X)[:, 1]

    @cached_property
    def _roc(self) -> tuple[np.ndarray, np.ndarray]:
        fpr, tpr, _ = roc_curve(self.data.val.y, self.scores)
        return fpr, tpr

    @property
    def fpr(self) -> np.ndarray:
        return self._roc[0]

    @property
    def tpr(self) -> np.ndarray:
        return self._roc[1]

    @cached_property
    def auc(self) -> float:
        return roc_auc_score(self.data.val.y, self.scores)

In [ ]:
class GraphRenderer:
    """Rendering utilities for model evaluation output."""

    COLORS: ClassVar[list[str]] = [
        "black", "steelblue", "darkorange", "seagreen",
        "crimson", "mediumpurple", "saddlebrown", "deeppink",
    ]

    @staticmethod
    def plot_roc_curves(models: list[Model]) -> None:
        """Plot ROC curves for a list of models, skipping unimplemented ones.

        Args:
            models: Model instances to evaluate. Models whose _build_search raises
                NotImplementedError are shown as dotted legend entries without a curve.
        """
        fig, ax = plt.subplots(figsize=(8, 7))
        for model, color in zip(models, GraphRenderer.COLORS):
            try:
                ax.plot(model.fpr, model.tpr, lw=2, color=color,
                        label=f"{model.label}   AUC = {model.auc:.3f}")
            except NotImplementedError:
                ax.plot([], [], lw=2, color=color, ls=":",
                        label=f"{model.label}   (not yet implemented)")
        ax.plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curves — All Models")
        ax.legend(loc="lower right", fontsize=8)
        plt.tight_layout()
        plt.show()

### Tyler Kouri

`NaiveBaseline` is implemented below. For `GradientBoostingModel`, implement `_build_search` to return a `RandomizedSearchCV` wrapping `GradientBoostingClassifier` with a search over `n_estimators`, `max_depth`, `learning_rate`, and `subsample`.

#### Naive Benchmark

The naive model ignores all features and predicts a single constant probability for every sample — the proportion of serious delinquencies in the training set. This is the best prediction available without using any features, making it the appropriate floor for every subsequent model. Under ROC AUC the benchmark scores exactly 0.5 by construction, since a constant vector imposes no ranking on the validation set. Any model we build must clear this threshold, and we will verify each AUC improvement is statistically significant rather than attributable to sampling variation.

In [ ]:
class NaiveBaseline(Model):
    """Constant predictor equal to the training base rate."""

    label: ClassVar[str] = "Naive baseline"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> None:
        return None

    @cached_property
    def scores(self) -> np.ndarray:
        return np.full(len(self.data.val.y), float(np.mean(self.data.train.y)))



In [ ]:
class GradientBoostingModel(Model):
    """Gradient Boosting on Branch A — raw, unscaled features.

    Suggested estimator : GradientBoostingClassifier(random_state=SEED)
    Suggested search    : n_estimators, max_depth, learning_rate, subsample
    """

    label: ClassVar[str] = "Gradient Boosting"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        raise NotImplementedError

## Support Vector Machine

**Authored by:** Stephen Wallen

SVM finds the maximum-margin separating hyperplane in feature space and is therefore sensitive to feature scale — features with large numeric ranges dominate the margin and suppress the contribution of smaller ones.

### Theory

A Support Vector Machine solves the soft-margin optimisation problem:

$$\min_{w,b,\xi}\;\frac{1}{2}\|w\|^2 + C\sum_{i=1}^n \xi_i \quad\text{subject to}\quad y_i\bigl(w^\top\phi(x_i)+b\bigr)\geq 1-\xi_i,\;\xi_i\geq 0$$

$w$ is the normal to the separating hyperplane; $\xi_i$ are slack variables permitting misclassification; $C$ controls the trade-off between margin width and training error. As $C \to \infty$ the slack terms are eliminated, recovering the hard-margin SVM.

### Hypothesis

We evaluate three SVM configurations in order of increasing sophistication, isolating the contribution of each design choice:

1. **Hard margin, Branch A (raw)** — Very large $C$ on unscaled features. Expected worst SVM result: features like `MonthlyIncome` (range 0–100k+) dominate the margin, and the rigid decision boundary overfits noise with no slack tolerance.
2. **Hard margin, Branch B (scaled)** — Same large $C$ but with standardised features. Removes the scale bias; expected to outperform variant 1.
3. **Soft margin, Branch B (scaled)** — Tuned $C$, $\gamma$, and kernel via random search. Expected best result: scale-corrected features plus a data-driven slack tolerance.

Two base classes are implemented below. `SVMHard` approximates the hard-margin SVM by setting $C = 10^6$ and fitting directly — it overrides `estimator` rather than using the search pipeline. `SVMSoft` returns a `RandomizedSearchCV` from `_build_search` and lets the base class handle tuning. Three branch subclasses select the preprocessing branch via `data`.

In [ ]:
from sklearn.svm import SVC
from scipy.stats import loguniform

SVM_HARD_C: float = 1e6   # approximates hard margin (C → ∞)

SVM_PARAMS: dict = {
    "C":      loguniform(1e-2, 1e3),
    "gamma":  loguniform(1e-4, 1e1),
    "kernel": ["rbf", "linear"],
}

In [ ]:
class SVMHard(Model):
    """Hard-margin SVM approximation (C=SVM_HARD_C, RBF kernel).

    Overrides estimator directly — no RandomizedSearchCV.
    Subclasses override label and data to select the preprocessing branch.
    """

    def _build_search(self) -> None:
        return None

    @cached_property
    def estimator(self) -> SVC:
        idx = np.random.RandomState(SEED).choice(
            len(self.data.train.X), size=min(TUNE_SAMPLE, len(self.data.train.X)), replace=False,
        )
        svc = SVC(
            C=SVM_HARD_C,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=SEED,
            cache_size=2000,
        )
        svc.fit(self.data.train.X.iloc[idx], self.data.train.y.iloc[idx])
        return svc

In [ ]:
class SVMSoft(Model):
    """Soft-margin SVM with tuned C, gamma, and kernel via RandomizedSearchCV.

    Subclasses override label and data to select the preprocessing branch.
    """

    def _build_search(self) -> RandomizedSearchCV:
        return RandomizedSearchCV(
            SVC(probability=True, class_weight="balanced", random_state=SEED, cache_size=2000),
            param_distributions=SVM_PARAMS,
            n_iter=TUNE_ITER,
            scoring="roc_auc",
            cv=TUNE_FOLDS,
            random_state=SEED,
            n_jobs=-1,
        )

In [ ]:
class SVMHardRaw(SVMHard):
    label: ClassVar[str] = "SVM — Hard margin, Raw"
    data:  ClassVar[Data] = Data.CLEAN


class SVMHardScaled(SVMHard):
    label: ClassVar[str] = "SVM — Hard margin, Scaled"
    data:  ClassVar[Data] = Data.SCALED


class SVMSoftScaled(SVMSoft):
    label: ClassVar[str] = "SVM — Soft margin, Scaled"
    data:  ClassVar[Data] = Data.SCALED

### Results

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves([SVMHardRaw(), SVMHardScaled(), SVMSoftScaled()])

### Conclusion

The hypothesis was partially confirmed: the soft-margin model on scaled features achieved the highest AUC, while the hard-margin results were inverted — the raw-feature variant outperformed the scaled variant, contrary to expectation. This inversion is likely a consequence of the hard-margin constraint itself: with $C 	o \infty$, high-magnitude features like `MonthlyIncome` dominate the RBF kernel in the raw space in a way that happens to be useful, whereas in the scaled space the equally-weighted features produce a harder separation problem with more support vectors. The key takeaway is that feature scaling and soft-margin regularisation are complementary — scaling is only reliably beneficial when $C$ is also tuned.

### Yuxuan Huang

Implement `_build_search` for both models below. For `LogisticRegressionModel`, return a `RandomizedSearchCV` wrapping `LogisticRegression` with a search over `C`, `penalty`, and `solver`. For `MLPModel`, wrap `MLPClassifier` with a search over `hidden_layer_sizes`, `alpha`, `learning_rate_init`, and `activation`.

In [ ]:
class LogisticRegressionModel(Model):
    """Logistic Regression on Branch B — standardised features.

    Suggested estimator : LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    Suggested search    : C (log-uniform), penalty, solver
    """

    label: ClassVar[str] = "Logistic Regression"
    data:  ClassVar[Data] = Data.SCALED

    def _build_search(self) -> RandomizedSearchCV:
        raise NotImplementedError

In [ ]:
class MLPModel(Model):
    """Multi-layer Perceptron on Branch B — standardised features.

    Suggested estimator : MLPClassifier(random_state=SEED, max_iter=300)
    Suggested search    : hidden_layer_sizes, alpha, learning_rate_init, activation
    """

    label: ClassVar[str] = "MLP"
    data:  ClassVar[Data] = Data.SCALED

    def _build_search(self) -> RandomizedSearchCV:
        raise NotImplementedError

### Aislinn O'Connell

Implement `_build_search` for both tree models below. For `RandomForestModel`, return a `RandomizedSearchCV` wrapping `RandomForestClassifier` with a search over `n_estimators`, `max_depth`, `min_samples_split`, and `max_features`. For `DecisionTreeModel`, wrap `DecisionTreeClassifier` with a search over `max_depth`, `min_samples_split`, `min_samples_leaf`, and `criterion`.

### Decision Trees

**Authored by:** Aislinn O'Connell

Decision trees are a machine learning algorithm that utilizes a tree where input data traverses through decision nodes that evaluate the data in a binary manner, greater than or less than. 

### Theory

The CART (Classification and Regression Tree) algorithm trains the decision trees. Initially, the dataset is split into two subsets with a single feature, k, and a threshold $t_k$. The algorithm searches for a pair ($k, t_k$), that produces the purest subset, which is evaluated by the Gini Impurity Equation. The Gini impurity of the node informs the user whether all training instances it applies to belong in the same class. A "pure" Gini score is 0. 

Gini Impurity Equation: 
$$ G_i = 1 - \sum_{k=1}^{n} p_{i,k}^2 $$

This process of splitting the dataset into two and finding the purest subset, is repeated on the split subsets. This recursive process will continue until the maximum depth is achieved, or it cannot be split in a way that will reduce the Gini Impurity score. 

CART Cost Function for Classification:
$$ J(k, t_k) = \frac{m_{left}}{m}G_{left} + \frac{m_{right}}{m}G_{right} $$

where 

 \begin{cases} 
      G_{left/right} \text{ measures the impurity of the left/right subset}\\
      m_{left/right} \text{ number of instances in left/right subset} \\
      m = m_{left} + m_{right}
\end{cases}

### Hypothesis

Decision Trees should outperform the naive baseline due to their ability to recognize nonlinear relationships, and interactions between variables. 

### Results

Decision trees have high variance, so changing hyperparameters can create substantially different models. Using RandomizedSearchCV(), random parameter combinations were tested and evaluated using cross-validation with TUNE_FOLDS folds over TUNE_ITER iterations, and the model with the highest average ROC AUC score was selected. The highest average ROC AUC score is **0.7926** for this model, which is considered a strong score for an imbalanced credit dataset, and the parameters chosen were: 

Optimized Decision Tree Model: 
DecisionTreeClassifier(class_weight='balanced', criterion='log_loss',max_depth=5, min_samples_leaf=11,      min_samples_split=25,random_state=42)


In [ ]:
class DecisionTreeModel(Model):
    """Decision Tree on Branch A — raw, unscaled features.

    Suggested estimator : DecisionTreeClassifier(class_weight="balanced", random_state=SEED)
    Suggested search    : max_depth, min_samples_split, min_samples_leaf, criterion
    """

    label: ClassVar[str] = "Decision Tree"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        #raise NotImplementedError
        estimator = DecisionTreeClassifier(
            class_weight="balanced",
            random_state=SEED,
        )

        param_distributions = {
            "max_depth": [None, 3, 5, 10, 20, 30, 50],
            "min_samples_split": randint(2, 51),
            "min_samples_leaf": randint(1, 21),
            "criterion": ["gini", "entropy", "log_loss"],
        }

        return RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_distributions,
            n_iter=TUNE_ITER,
            cv=TUNE_FOLDS,
            scoring="roc_auc",
            random_state=SEED,
            n_jobs=-1,
        )
    
decision_tree = DecisionTreeModel()

print("Decision Tree ROC AUC Score =", decision_tree.auc)
print("Best Parameters for the Decision Tree: ", decision_tree.search.best_estimator_)

### Random Forest

**Authored by:** Aislinn O'Connell

The Random Forest is a machine learning algorithm that relies on an ensemble of decision trees that searches for the best feature among a random subset of features.

### Theory

The ensemble of decision trees within a Random Forest are usually trained with the bagging (Bootstrap Aggregating)  algorithm. Bagging repeatedly generates bootstrap samples by sampling observations from the training set at random with replacement, the same observation(row) can be used multiple times within a single tree's training sample. The Random Forest selects only a random subset of features at each split, instead of comparing every feature, ensuring that the trees are not overly customized to the data, and do not form strong correlations. Each individual decision tree generates a prediction independently, and the prediction from the Random Forest is found through majority voting across all trees 

Random Forests Random Forest improves upon a single Decision Tree by reducing variance while preserving low bias. Individual trees may overfit the training data, but averaging predictions across many trees produces a more stable and generalizable model.

### Hypothesis

The Random Forest algorithm will outperform the naive baseline, as well as the singular Decision Tree model. This is because Random Forest combines the predictions from multiple random Decision Trees, and adds randommess into the process through bootstrap sampling and random feature selection. This will reduce overfitting and improve performance overall. It is predicted that the Random Forest will have a higher ROC AUC score when predicting serious delinquency within two years than the Decision Tree and Naive Baseline. 

### Results

Random Forest was tuned using RandomizedSearchCV(), where random parameter combinations were evaluated using cross-validation with TUNE_FOLDS folds over TUNE_ITER iterations. Random Forest outperformed the naive baseline, but barely outperformed the Decision Tree model. The ROC AUC score for the Random Forest is **0.7985**, compared to the score of **0.7926** for the Decision Tree.

The Random Forest and Decision Tree models, despite having similar ROC AUC scores, had different parameters. The Decision Tree minimized overfitting by staying smaller and avoiding additional complexity with a max depth of 5, a larger minimum leaf size of 11, and a minimum split requirement of 25. 

Comparatively, the Random Forest had a maximum depth of 20 and a minimum split threshold of 16, which allowed individual decision trees to become more complex. This reduces overfitting by reducing variance through averaging predictions across multiple flexible decision trees.

Despite using different methodologies for reducing overfitting, the Random Forest and Decision Tree models achieved similar results.


Best Parameters for the Random Forest:  RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_split=16, n_estimators=206, n_jobs=-1,
                       random_state=42)




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

class RandomForestModel(Model):
    """Random Forest on Branch A — raw, unscaled features.

    Suggested estimator : RandomForestClassifier(class_weight="balanced", random_state=SEED)
    Suggested search    : n_estimators, max_depth, min_samples_split, max_features
    """

    label: ClassVar[str] = "Random Forest"
    data:  ClassVar[Data] = Data.CLEAN

    def _build_search(self) -> RandomizedSearchCV:
        estimator = RandomForestClassifier(
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )

        param_distributions = {
            "n_estimators": randint(100, 601),
            "max_depth": [None, 5, 10, 20, 30, 50],
            "min_samples_split": randint(2, 21),
            "max_features": ["sqrt", "log2", None],
        }

        return RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_distributions,
            n_iter=TUNE_ITER,
            cv=TUNE_FOLDS,
            scoring="roc_auc",
            random_state=SEED,
            n_jobs=-1,
        )
    
rand_forest = RandomForestModel()

print("Random Forest ROC AUC Score =", rand_forest.auc)
print("Best Parameters for the Random Forest: ", rand_forest.search.best_estimator_)

### Conclusion for Decision Trees & Random Forest

Both the Decision Tree and Random Forest models achieved ROC AUC scores of roughly **0.80**, demonstrating strong predictive ability for identifying serious delinquency within two years. Although Random Forest typically reduces overfitting through bagging and the introduction of randomness, only a marginal improvement over the Decision Tree model was observed.

One possible explanation is that the preprocessing pipeline reduced noise and improved feature quality, allowing the simpler Decision Tree model to capture most of the predictive structure present in the dataset. This suggests that the additional complexity introduced by the Random Forest model provided only a limited increase in predictive performance.

Therefore, while Random Forest slightly outperformed the Decision Tree, the improvement was relatively small and may not justify the decrease in interpretability associated with ensemble methods for this particular dataset.

In [ ]:
MODELS: list[Model] = [
    NaiveBaseline(),
    SVMHardRaw(), SVMHardScaled(), SVMSoftScaled(),
    LogisticRegressionModel(),
    GradientBoostingModel(),
    RandomForestModel(),
    DecisionTreeModel(),
    MLPModel(),
]

In [ ]:
#| echo: false
GraphRenderer.plot_roc_curves(MODELS)